In [1]:
import os
import sys
import pandas as pd
import json
import pickle

In [2]:
# load subquery_plans_joblight.json
dict_subqueries = {}
with open('subquery_plans_joblight.json') as f:
    dict_subqueries = json.load(f)

In [3]:
# read files in query folder
def read_files(folder):
    files = []
    for file in os.listdir(folder):
        if file.endswith(".txt"):
            files.append(file)
    return files

# read queries from files
def read_queries(folder, files, subqueries):
    dict_queries = {}
    dict_subqueries = {}
    query_counter = 0
    
    for file in files:
        with open(folder + file, 'r') as f:
            q = f.read()
            q = q.replace('\n', ' ')
            #q = q + ";"
            dict_queries[str(query_counter)] = q
            
            subqueries_q = subqueries[str(query_counter)]
            
            dict_subqueries[str(query_counter)] = []
            # iteratate over values in subqueries_q
            for q in subqueries_q.values():
                outer_ = q['outer_relid']
                outer_ = outer_.split(" ")
                # sorrt outer_ to get the correct order
                outer_ = sorted(outer_)
                outer_ = " ".join(outer_)
                tuple_ = (q['inner_relid'], outer_)
                dict_subqueries[str(query_counter)].append(tuple_)
            
            query_counter += 1
            

    return (dict_queries, dict_subqueries)



In [4]:
dict_qs = read_queries('joblight/', read_files('joblight/'), dict_subqueries)


In [30]:
dict_qs[0]

{'0': 'SELECT COUNT(*)  FROM movie_companies AS mc, title AS t, movie_info_idx AS mi_idx   WHERE t.id=mc.movie_id  AND t.id=mi_idx.movie_id  AND mi_idx.info_type_id=112  AND mc.company_type_id=2 ',
 '1': 'SELECT COUNT(*)  FROM movie_companies AS mc, title AS t, movie_info_idx AS mi_idx  WHERE t.id=mc.movie_id AND t.id=mi_idx.movie_id  AND mi_idx.info_type_id=113  AND mc.company_type_id=2 AND  t.production_year>2005  AND t.production_year<2010 ',
 '2': 'SELECT COUNT(*)  FROM movie_companies AS mc, title AS t, movie_info_idx AS mi_idx  WHERE t.id=mc.movie_id AND t.id=mi_idx.movie_id AND mi_idx.info_type_id=112 AND mc.company_type_id=2 AND t.production_year>2010 ',
 '3': 'SELECT COUNT(*) FROM movie_companies AS mc,title AS t,movie_info_idx AS mi_idx WHERE t.id=mc.movie_id AND t.id=mi_idx.movie_id AND mi_idx.info_type_id=113 AND mc.company_type_id=2 AND t.production_year>2000 ',
 '4': 'SELECT COUNT(*) FROM movie_companies AS mc,title AS t,movie_keyword AS mk WHERE t.id=mc.movie_id AND t.id

In [19]:
count_subs = 0
for subs in dict_qs[1]:
    #print(len(dict_qs[1][subs]))
    count_subs += len(dict_qs[1][subs])
    
    
count_subs

696

In [29]:
dict_qs[1]

{'0': [('t', 'mc'), ('mi_idx', 'mc'), ('mi_idx', 't'), ('mi_idx', 'mc t')],
 '1': [('t', 'mc'), ('mi_idx', 'mc'), ('mi_idx', 't'), ('mi_idx', 'mc t')],
 '2': [('t', 'mc'), ('mi_idx', 'mc'), ('mi_idx', 't'), ('mi_idx', 'mc t')],
 '3': [('t', 'mc'), ('mi_idx', 'mc'), ('mi_idx', 't'), ('mi_idx', 'mc t')],
 '4': [('t', 'mc'), ('mk', 'mc'), ('mk', 't'), ('mk', 'mc t')],
 '5': [('mi', 't'), ('mk', 't'), ('mk', 'mi'), ('mk', 'mi t')],
 '6': [('mi', 't'), ('mk', 't'), ('mk', 'mi'), ('mk', 'mi t')],
 '7': [('mi', 't'), ('mk', 't'), ('mk', 'mi'), ('mk', 'mi t')],
 '8': [('mi_idx', 't'), ('mk', 't'), ('mk', 'mi_idx'), ('mk', 'mi_idx t')],
 '9': [('mi_idx', 't'), ('mk', 't'), ('mk', 'mi_idx'), ('mk', 'mi_idx t')],
 '10': [('mi_idx', 't'), ('mk', 't'), ('mk', 'mi_idx'), ('mk', 'mi_idx t')],
 '11': [('mi', 't'), ('mc', 't'), ('mc', 'mi'), ('mc', 'mi t')],
 '12': [('mi', 't'), ('mc', 't'), ('mc', 'mi'), ('mc', 'mi t')],
 '13': [('mi', 't'), ('mc', 't'), ('mc', 'mi'), ('mc', 'mi t')],
 '14': [('t', 'm

In [10]:
sub

'9'

In [45]:
with open('dict_sql_imdb_jobjoin.pkl', 'wb') as f:
    pickle.dump(dict_qs, f)